# Latin-Masking Pipeline (Module API)

This notebook runs the latin-masking pipeline step by step using the module-based API.

## Pipeline Steps:
1. Sentence splitting
2. Generate common adverbs
3. -que splitting
4. POS tagging and masking

## Step 1: Setup and Imports

In [ ]:
from collections import Counter
from pathlib import Path

# Import latin-masking modules
from latin_masking import process_file_with_cache
from latin_masking.adverbs import (
    collect_adverbs,
    generate_adverb_list,
    load_adverb_list,
    normalize_adverb_counts,
    save_adverb_list,
)
from latin_masking.clitics import split_que_blacklist
from latin_masking.conllu import parse_conllu
from latin_masking.mask import two_pass_mask
from latin_masking.sentences import split_sentences

# Configuration
INPUT_DIR = Path("/Users/ben/code/Liber-Regum/Lexical analysis/data")
OUTPUT_DIR = INPUT_DIR
MODEL = "latin-evalatin24-240520"
ADVERB_THRESHOLD = 200

print("Imports complete. Ready to run pipeline.")

Imports complete. Ready to run pipeline.


## Step 2: Sentence Splitting

Split raw text files into sentences using the la_senter spaCy model.

In [3]:
# Find raw .txt files (excluding already processed files)
txt_files = sorted([f for f in INPUT_DIR.glob("*.txt") if "_sentences" not in f.name])
print(f"Found {len(txt_files)} files to process:")
for f in txt_files:
    print(f"  - {f.name}")

Found 34 files to process:
  - AE_bhc.txt
  - ASt_troilus.txt
  - BB_adelae.txt
  - BBi_speculum.txt
  - BI_reynardus.txt
  - BM_regum.txt
  - BS_mathematicus.txt
  - EM_mahumeti.txt
  - H?_cilr.txt
  - HA_pentateuchum.txt
  - HM_gestis.txt
  - HW_hortus.txt
  - H_carmina.txt
  - H_mysterio.txt
  - IS_entheticus.txt
  - M_ars.txt
  - M_epistulae.txt
  - M_tobias.txt
  - NLC_miracula.txt
  - NLC_speculum.txt
  - PDE_rebussiculis.txt
  - PR_aurora.txt
  - QS_alexandri.txt
  - RDV_paulino.txt
  - RL_anselmi.txt
  - RT_epistulae.txt
  - RT_memorabilibus.txt
  - RT_miracula.txt
  - SAC_excidio.txt
  - SR_draco.txt
  - U1_pyramo.txt
  - U2_ysengrimus.txt
  - U3_guiardinus.txt
  - X_liberregum.txt


In [4]:
# Process each file
for txt_file in txt_files:
    with open(txt_file, "r", encoding="utf-8") as f:
        text = f.read()

    sentences = split_sentences(text)

    output_path = OUTPUT_DIR / f"{txt_file.stem}_sentences.txt"
    with open(output_path, "w", encoding="utf-8") as f:
        for sent in sentences:
            f.write(sent + "\n")

    print(f"Wrote {len(sentences)} sentences to {output_path.name}")

Wrote 408 sentences to AE_bhc_sentences.txt
Wrote 3495 sentences to ASt_troilus_sentences.txt
Wrote 697 sentences to BB_adelae_sentences.txt
Wrote 981 sentences to BBi_speculum_sentences.txt
Wrote 762 sentences to BI_reynardus_sentences.txt
Wrote 610 sentences to BM_regum_sentences.txt
Wrote 392 sentences to BS_mathematicus_sentences.txt
Wrote 529 sentences to EM_mahumeti_sentences.txt
Wrote 789 sentences to H?_cilr_sentences.txt
Wrote 498 sentences to HA_pentateuchum_sentences.txt
Wrote 3471 sentences to HM_gestis_sentences.txt
Wrote 4543 sentences to HW_hortus_sentences.txt
Wrote 339 sentences to H_carmina_sentences.txt
Wrote 310 sentences to H_mysterio_sentences.txt
Wrote 959 sentences to IS_entheticus_sentences.txt
Wrote 393 sentences to M_ars_sentences.txt
Wrote 889 sentences to M_epistulae_sentences.txt
Wrote 1159 sentences to M_tobias_sentences.txt
Wrote 1551 sentences to NLC_miracula_sentences.txt
Wrote 2001 sentences to NLC_speculum_sentences.txt
Wrote 864 sentences to PDE_reb

## Step 3: Generate Common Adverbs

Process sentences through UDPipe and collect adverbs to build a common adverbs list.

In [5]:
# Get all _sentences.txt files
sentences_files = sorted(INPUT_DIR.glob("*_sentences.txt"))
print(f"Found {len(sentences_files)} sentence files:")
for f in sentences_files:
    print(f"  - {f.name}")

Found 34 sentence files:
  - AE_bhc_sentences.txt
  - ASt_troilus_sentences.txt
  - BB_adelae_sentences.txt
  - BBi_speculum_sentences.txt
  - BI_reynardus_sentences.txt
  - BM_regum_sentences.txt
  - BS_mathematicus_sentences.txt
  - EM_mahumeti_sentences.txt
  - H?_cilr_sentences.txt
  - HA_pentateuchum_sentences.txt
  - HM_gestis_sentences.txt
  - HW_hortus_sentences.txt
  - H_carmina_sentences.txt
  - H_mysterio_sentences.txt
  - IS_entheticus_sentences.txt
  - M_ars_sentences.txt
  - M_epistulae_sentences.txt
  - M_tobias_sentences.txt
  - NLC_miracula_sentences.txt
  - NLC_speculum_sentences.txt
  - PDE_rebussiculis_sentences.txt
  - PR_aurora_sentences.txt
  - QS_alexandri_sentences.txt
  - RDV_paulino_sentences.txt
  - RL_anselmi_sentences.txt
  - RT_epistulae_sentences.txt
  - RT_memorabilibus_sentences.txt
  - RT_miracula_sentences.txt
  - SAC_excidio_sentences.txt
  - SR_draco_sentences.txt
  - U1_pyramo_sentences.txt
  - U2_ysengrimus_sentences.txt
  - U3_guiardinus_sentenc

In [ ]:
# Cache directory for raw UDPipe responses
RAW_CACHE_DIR = INPUT_DIR / "udpipe_cache_raw"
RAW_CACHE_DIR.mkdir(exist_ok=True)

# Collect adverbs from all sentence files
all_adverbs: Counter[str] = Counter()

for sent_file in sentences_files:
    # Process with automatic caching
    response = process_file_with_cache(
        sent_file,
        MODEL,
        cache_dir=RAW_CACHE_DIR,
        presegmented=True,
        raw=True,
        unsafe_certs_ok=True,
    )

    if response:
        frames, _ = parse_conllu(response)
        advs = collect_adverbs(frames)
        all_adverbs.update(advs)
        print(f"  {sent_file.name}: found {len(advs)} adverbs")

print(f"\nTotal unique adverbs: {len(all_adverbs)}")

  AE_bhc_sentences.txt: found 71 adverbs
  ASt_troilus_sentences.txt: found 348 adverbs
  BB_adelae_sentences.txt: found 153 adverbs
  BBi_speculum_sentences.txt: found 185 adverbs
  BI_reynardus_sentences.txt: found 169 adverbs
  BM_regum_sentences.txt: found 101 adverbs
  BS_mathematicus_sentences.txt: found 110 adverbs
  EM_mahumeti_sentences.txt: found 169 adverbs
  H?_cilr_sentences.txt: found 116 adverbs
  HA_pentateuchum_sentences.txt: found 88 adverbs
  HM_gestis_sentences.txt: found 233 adverbs
  HW_hortus_sentences.txt: found 234 adverbs
  H_carmina_sentences.txt: found 94 adverbs
  H_mysterio_sentences.txt: found 95 adverbs
  IS_entheticus_sentences.txt: found 150 adverbs
  M_ars_sentences.txt: found 30 adverbs
  M_epistulae_sentences.txt: found 73 adverbs
  M_tobias_sentences.txt: found 75 adverbs
  NLC_miracula_sentences.txt: found 212 adverbs
  NLC_speculum_sentences.txt: found 259 adverbs
  PDE_rebussiculis_sentences.txt: found 143 adverbs
  PR_aurora_sentences.txt: foun

In [ ]:
# Normalize and get top adverbs
normalized = normalize_adverb_counts(all_adverbs)
top_adverbs = generate_adverb_list(normalized, ADVERB_THRESHOLD)
# Get all adverbs without threshold
all_adverbs_list = generate_adverb_list(normalized, None)
print(f"Top {len(top_adverbs)} adverbs:")
for adv, count in top_adverbs[:20]:
    print(f"  {adv}\t{count}")
print("  ...")

Top 200 adverbs:
  sic	2607
  iam	1207
  tamen	1036
  inde	1019
  hinc	904
  ergo	864
  tunc	822
  nunc	763
  semper	718
  hic	612
  bene	499
  ibi	479
  simul	461
  magis	447
  nimis	420
  cur	391
  unde	373
  tam	368
  modo	364
  uix	363
  ...


In [ ]:
# Save adverb list
adverbs_path = Path(".") / "common_adverbs.txt"
all_adverbs_path = Path(".") / "all_adverbs.txt"
save_adverb_list(top_adverbs, adverbs_path)
save_adverb_list(all_adverbs_list, all_adverbs_path)
print(f"Saved adverbs to {adverbs_path} and {all_adverbs_path}")

Saved adverbs to common_adverbs.txt and all_adverbs.txt


## Step 4: -que Splitting

Split -que enclitics using the blacklist approach. Common adverbs are automatically
protected from splitting.

In [10]:
# Load common adverbs for masking (these will be protected from -que splitting)
common_adverbs = load_adverb_list(adverbs_path, ADVERB_THRESHOLD)
print(f"Loaded {len(common_adverbs)} common adverbs for masking")

# Process each _sentences.txt file
for sent_file in sentences_files:
    with open(sent_file, "r", encoding="utf-8") as f:
        text = f.read()

    # Split -que, automatically protecting common adverbs from splitting
    new_text, count = split_que_blacklist(text, common_adverbs=common_adverbs)

    output_path = OUTPUT_DIR / f"{sent_file.stem}.quesplit.txt"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(new_text)

    print(f"{sent_file.name}: made {count} -que replacements")

Loaded 200 common adverbs for masking
AE_bhc_sentences.txt: made 89 -que replacements
ASt_troilus_sentences.txt: made 450 -que replacements
BB_adelae_sentences.txt: made 174 -que replacements
BBi_speculum_sentences.txt: made 186 -que replacements
BI_reynardus_sentences.txt: made 300 -que replacements
BM_regum_sentences.txt: made 158 -que replacements
BS_mathematicus_sentences.txt: made 205 -que replacements
EM_mahumeti_sentences.txt: made 105 -que replacements
H?_cilr_sentences.txt: made 144 -que replacements
HA_pentateuchum_sentences.txt: made 192 -que replacements
HM_gestis_sentences.txt: made 178 -que replacements
HW_hortus_sentences.txt: made 42 -que replacements
H_carmina_sentences.txt: made 78 -que replacements
H_mysterio_sentences.txt: made 77 -que replacements
IS_entheticus_sentences.txt: made 255 -que replacements
M_ars_sentences.txt: made 47 -que replacements
M_epistulae_sentences.txt: made 102 -que replacements
M_tobias_sentences.txt: made 44 -que replacements
NLC_miracula_s

## Step 5: POS Tagging and Masking

Process quesplit files through UDPipe and apply POS masking. Cached responses are
saved to avoid re-processing.

In [ ]:
# Get all _sentences.quesplit.txt files
quesplit_files = sorted(INPUT_DIR.glob("*_sentences.quesplit.txt"))
print(f"Found {len(quesplit_files)} quesplit files:")
for f in quesplit_files:
    print(f"  - {f.name}")

# Cache directory for UDPipe responses
CACHE_DIR = INPUT_DIR / "udpipe_cache_quesplit"
CACHE_DIR.mkdir(exist_ok=True)

# Process each quesplit file
for qs_file in quesplit_files:
    # Process with automatic caching
    response = process_file_with_cache(
        qs_file,
        MODEL,
        cache_dir=CACHE_DIR,
        presegmented=True,
        raw=True,
        unsafe_certs_ok=True,
    )

    if response:
        frames, _ = parse_conllu(response)
        masked = two_pass_mask(frames, common_adverbs=common_adverbs)

        output_path = OUTPUT_DIR / f"{qs_file.stem}.masked.txt"
        with open(output_path, "w", encoding="utf-8") as f:
            f.write("\n".join(masked))

        print(f"{qs_file.name}: wrote {len(masked)} masked sentences")

Found 34 quesplit files:
  - AE_bhc_sentences.quesplit.txt
  - ASt_troilus_sentences.quesplit.txt
  - BB_adelae_sentences.quesplit.txt
  - BBi_speculum_sentences.quesplit.txt
  - BI_reynardus_sentences.quesplit.txt
  - BM_regum_sentences.quesplit.txt
  - BS_mathematicus_sentences.quesplit.txt
  - EM_mahumeti_sentences.quesplit.txt
  - H?_cilr_sentences.quesplit.txt
  - HA_pentateuchum_sentences.quesplit.txt
  - HM_gestis_sentences.quesplit.txt
  - HW_hortus_sentences.quesplit.txt
  - H_carmina_sentences.quesplit.txt
  - H_mysterio_sentences.quesplit.txt
  - IS_entheticus_sentences.quesplit.txt
  - M_ars_sentences.quesplit.txt
  - M_epistulae_sentences.quesplit.txt
  - M_tobias_sentences.quesplit.txt
  - NLC_miracula_sentences.quesplit.txt
  - NLC_speculum_sentences.quesplit.txt
  - PDE_rebussiculis_sentences.quesplit.txt
  - PR_aurora_sentences.quesplit.txt
  - QS_alexandri_sentences.quesplit.txt
  - RDV_paulino_sentences.quesplit.txt
  - RL_anselmi_sentences.quesplit.txt
  - RT_epistu

## Step 6: View Results

Compare original, sentences, and masked output.

In [13]:
# Show example output
example_file = txt_files[0]  # First input file
print(f"=== Original ({example_file.name}) ===")
with open(example_file, "r", encoding="utf-8") as f:
    original = f.read()
print(original[:500] + "..." if len(original) > 500 else original)

sentences_file = INPUT_DIR / f"{example_file.stem}_sentences.txt"
print(f"\n=== Sentences ({sentences_file.name}) ===")
with open(sentences_file, "r", encoding="utf-8") as f:
    sentences = f.readlines()
for i, sent in enumerate(sentences[:5]):
    print(f"{i+1}. {sent.strip()}")
print(f"... ({len(sentences)} total sentences)")

masked_file = INPUT_DIR / f"{example_file.stem}_sentences.quesplit.masked.txt"
print(f"\n=== Masked ({masked_file.name}) ===")
with open(masked_file, "r", encoding="utf-8") as f:
    masked = f.readlines()
for i, sent in enumerate(masked[:5]):
    print(f"{i+1}. {sent.strip()}")
print(f"... ({len(masked)} total sentences)")

=== Original (AE_bhc.txt) ===
Ante dies omnes mundi fuit omnis in uno
	Machina moment facta iubente Deo.
Sed tunc nec celum, nec terra, nec unda, nec aer
	Ornatus habuit quos habet, unde nitet.
Vnda tegit terram, tegit aera, sic elementa
	Hec tria miscentur efficiuntque chaos.
Hec polus empireus superat, ternos ter in ista
	Angelicos cetus collocat arce Deus.
Lumine uinutum cunctis his angelus unus
	Prelucens, dictus Lucifer inde fuit.
Hunc tumor et multos a ce tnrdit ad ima;
	Qui fuerant humiles promeruere statum.
Nec possu...

=== Sentences (AE_bhc_sentences.txt) ===
1. Ante dies omnes mundi fuit omnis in uno Machina moment facta iubente Deo.
2. Sed tunc nec celum, nec terra, nec unda, nec aer Ornatus habuit quos habet, unde nitet.
3. Vnda tegit terram, tegit aera, sic elementa Hec tria miscentur efficiuntque chaos.
4. Hec polus empireus superat, ternos ter in ista Angelicos cetus collocat arce Deus.
5. Lumine uinutum cunctis his angelus unus Prelucens, dictus Lucifer inde fuit.
... 